# pi-swe-bench — results dashboard

Reads every run archive under `results/runs/*.json` (written by `run_bench.py`) and
turns them into pass@1 tables, tier curves, the auto-vs-guided diagnosis gap, and
run-over-run tracking.

**Usage:** run a benchmark first, then run all cells here. Nothing in this notebook
launches a model — it only reads JSON, so it's fast and safe to re-run. To kick off
a run from here without blocking the kernel, use a terminal, or:

```python
# import subprocess; subprocess.Popen(["python", "run_bench.py", "--guidance", "standard", "--samples", "3"])
```

In [ ]:
%matplotlib inline
import glob, json, os
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = "results"

def load_runs(runs_dir=None, flat_glob=None):
    """Load every run archive into one tidy DataFrame (one row per model x task).

    Prefers results/runs/*.json (full history + metadata). Falls back to the flat
    results_*.json convenience files (latest only) if no archives exist yet."""
    runs_dir = runs_dir or os.path.join(RESULTS, "runs")
    flat_glob = flat_glob or os.path.join(RESULTS, "results_*.json")
    files = sorted(glob.glob(os.path.join(runs_dir, "*.json")))
    rows = []
    if files:
        for f in files:
            d = json.load(open(f)); meta = d.get("meta", {})
            for r in d.get("results", []):
                r = dict(r)
                r.setdefault("run_id", meta.get("run_id"))
                r.setdefault("git_sha", meta.get("git_sha"))
                r.setdefault("timestamp", meta.get("timestamp"))
                r["samples"] = meta.get("samples")
                rows.append(r)
    else:
        for f in sorted(glob.glob(flat_glob)):
            rows += json.load(open(f))
    df = pd.DataFrame(rows)
    if not df.empty and "timestamp" in df:
        df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df

def latest(df):
    """Keep only the most recent run per (model, guidance)."""
    keep = df.groupby(["model", "guidance"])["run_id"].transform("max") == df["run_id"]
    return df[keep]

df = load_runs()
assert not df.empty, "No results found. Run `python run_bench.py` first."
print(f"{len(df)} rows | {df.run_id.nunique()} run(s) | "
      f"models={list(df.model.unique())} | guidance={sorted(df.guidance.unique())}")
df.head()

## Latest snapshot — pass@1 by model × tier

Most recent run per (model, guidance). Set `GUIDANCE` to the level you care about.

In [ ]:
GUIDANCE = "standard"   # "auto" | "standard" | "guided"

L = latest(df)
snap = L[L.guidance == GUIDANCE]
by_tier = snap.pivot_table("pass_at_1", "model", "tier")
display(by_tier.round(2))

ax = by_tier.T.plot(marker="o", figsize=(6, 4))
ax.set_title(f"pass@1 by tier ({GUIDANCE})"); ax.set_xlabel("tier")
ax.set_ylabel("pass@1"); ax.set_ylim(0, 1.02); ax.grid(alpha=.3); plt.show()

print("overall pass@1:")
display(snap.groupby("model")["pass_at_1"].mean().round(3))

## Diagnosis vs fix — auto vs guided

`guided` names the buggy function; `auto` makes the agent find it. A large
`guided − auto` gap means the weakness is localization, not repair.

In [ ]:
ov = latest(df).pivot_table("pass_at_1", "model", "guidance")
if {"guided", "auto"} <= set(ov.columns):
    ov["diagnosis_gap"] = ov["guided"] - ov["auto"]
display(ov.round(2))

cols = [c for c in ("auto", "standard", "guided") if c in ov.columns]
if cols:
    ax = ov[cols].plot(kind="bar", figsize=(6, 4))
    ax.set_title("overall pass@1 by guidance"); ax.set_ylabel("pass@1")
    ax.set_ylim(0, 1.02); ax.grid(axis="y", alpha=.3)
    plt.xticks(rotation=0); plt.show()

## Run-over-run tracking

Overall pass@1 for each run over time, plus the change from the previous run at a
given guidance. This is what the `run_id` / `timestamp` / `git_sha` stamps buy you:
you can attribute a score change to a task edit or a config change instead of guessing.

In [ ]:
TRACK_GUIDANCE = "standard"

hist = (df[df.guidance == TRACK_GUIDANCE]
        .groupby(["timestamp", "run_id", "git_sha", "model"])["pass_at_1"]
        .mean().reset_index().sort_values("timestamp"))
display(hist.assign(pass_at_1=hist.pass_at_1.round(3)))

if hist.run_id.nunique() > 1:
    ax = hist.pivot_table("pass_at_1", "timestamp", "model").plot(marker="o", figsize=(7, 4))
    ax.set_title(f"overall pass@1 over runs ({TRACK_GUIDANCE})")
    ax.set_ylabel("pass@1"); ax.set_ylim(0, 1.02); ax.grid(alpha=.3); plt.show()

    piv = hist.pivot_table("pass_at_1", "model", "run_id")
    order = hist.drop_duplicates("run_id").sort_values("timestamp")["run_id"].tolist()
    last, prev = order[-1], order[-2]
    delta = (piv[last] - piv[prev]).rename(f"Δ ({prev} → {last})").round(3)
    print("change vs previous run:"); display(delta)
else:
    print("Only one run at this guidance — run again to see a trend.")

## Cost / latency

Average wall-clock seconds per task. With both models fully offloaded on your M4
these are comparable; a model that fails a tier *and* is slow there is often just
running out of turns — confirm in the artifacts before concluding it's weaker.

In [ ]:
lat = latest(df)
display(lat.pivot_table("avg_seconds", "model", "tier").round(0))

ax = lat.plot.scatter(x="avg_seconds", y="pass_at_1", c="tier",
                      colormap="viridis", figsize=(6, 4))
ax.set_title("pass@1 vs wall-clock, per task (latest)")
ax.set_ylim(-0.02, 1.02); ax.grid(alpha=.3); plt.show()

## Drill into failures

Rows the models missed in the latest runs, with the path to each attempt's
artifacts (`prompt.txt`, `solution.diff`, `pi.log`, `grade.log`) so you can see
*why* — a wrong fix, an unfixed bug, or the agent stopping early.

In [ ]:
fail = latest(df).query("pass_at_1 < 1").sort_values(["guidance", "tier", "model"])
print(f"{len(fail)} (model, task, guidance) cells below 100%:")
display(fail[["model", "guidance", "tier", "task", "pass_at_1"]].reset_index(drop=True))

if not fail.empty:
    r = fail.iloc[0]
    hits = glob.glob(os.path.join(RESULTS, "artifacts", r.guidance, r.model, f"*_{r.task}", "sample0"))
    if hits:
        print(f"\ninspect {r.model} / {r.task} / {r.guidance}:")
        for fn in ("prompt.txt", "solution.diff", "pi.log", "grade.log"):
            print("  ", os.path.join(hits[0], fn))
    else:
        print("\n(no on-disk artifacts for this row — loaded from a flat results file?)")